# 🧠 AIOS-Kernel-SLM: 15-Minute Turnkey Fine-Tuning Notebook
### Sovereign Operating System Intelligence (0.5B - 1.5B SLM)

**Objective:** Fine-tune `Qwen2.5-1.5B-Instruct` or `Qwen2.5-0.5B-Instruct` to create the proprietary **`AIOS-Kernel-SLM`**.

### ⚡ Why This Model Is Needed
- AIOS exposes **68 MCP tools** via `aiosh-mcp` (Rust).
- Passing all 68 full JSON schemas to generic models produces a **~12,000-token system prompt**.
- On consumer CPU (laptops/desktops), evaluating 12,000 tokens takes **3+ minutes per turn**.
- By fine-tuning this SLM on the synthetic AIOS corpus, tool schemas are **baked directly into the model's weights**.
- System prompt shrinks from 12,000 tokens to **~35 tokens** (99.7% reduction).
- Quantized in GGUF (`Q4_K_M`), the model uses **350MB - 980MB RAM** and runs at **50–100+ tokens/sec on CPU** with <1.5s total turnaround time!

## Step 1: Install High-Performance Fine-Tuning Libraries (Unsloth / TRL)

In [ ]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

## Step 2: Load Base Model & Enable 4-bit QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto detection (Float16 or Bfloat16)
load_in_4bit = True  # Use 4bit quantization to reduce memory usage

# Select model: Qwen/Qwen2.5-1.5B-Instruct or Qwen/Qwen2.5-0.5B-Instruct
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"[*] Loading model {model_id}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Apply LoRA adapters to all projection layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("[+] Model loaded and LoRA adapters configured!")

## Step 3: Load AIOS Synthetic Dataset (`aios_train.jsonl`)

In [ ]:
import json
from datasets import Dataset

# If running in Colab, upload aios_train.jsonl directly or fetch from repo
dataset_path = "aios_train.jsonl"

# Helper to format into Qwen2.5 ChatML with tool call formatting
def format_chatml(messages):
    text = ""
    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", [])
        text += f"<|im_start|>{role}\n"
        if content:
            text += f"{content}\n"
        if tool_calls:
            for tc in tool_calls:
                fn = tc.get("function", {})
                name = fn.get("name", "")
                args = fn.get("arguments", "{}")
                text += f"<tool_call>\n{{\"name\": \"{name}\", \"arguments\": {args}}}\n</tool_call>\n"
        text += "<|im_end|>\n"
    return text

samples = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        samples.append({"text": format_chatml(data["messages"])})

dataset = Dataset.from_list(samples)
print(f"[+] Loaded {len(dataset)} training examples.")
print("Sample snippet:\n", dataset[0]["text"][:350])

## Step 4: Execute SFT Training (Supervised Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        max_steps=300,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="aios_checkpoints",
    ),
)

print("[*] Training started...")
trainer_stats = trainer.train()
print("[+] Training finished successfully!")

## Step 5: Export to GGUF (`Q4_K_M`) for Consumer CPU

In [ ]:
# Export GGUF using Unsloth native fast quantization
print("[*] Exporting to GGUF Q4_K_M (optimal for laptop CPU)...\n")
model.save_pretrained_gguf("aios-kernel-1.5b-q4_k_m", tokenizer, quantization_method="q4_k_m")
print("[+] Export complete: aios-kernel-1.5b-q4_k_m.gguf is ready!")

## Step 6: Create Modelfile & Register in Ollama

Save this `Modelfile` alongside the `.gguf` file:
```dockerfile
FROM ./aios-kernel-1.5b-q4_k_m.gguf

SYSTEM """You are the AIOS Kernel Assistant, an S-rank autonomous operating system intelligence. You have direct control over the host Linux operating system through AIOS MCP tools. Always invoke the appropriate tools for system queries and management actions. Output concise markdown tables for structured data."""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.1
PARAMETER top_p 0.95
```

Register with Ollama:
```bash
ollama create aios-kernel -f Modelfile
python ai_agent.py --provider ollama --model aios-kernel
```